# Siamese BiLSTM v11 — Model Utama untuk LMS

Notebook ini melatih **satu model final** menggunakan seluruh data (original + sintetis).  
Model yang dihasilkan digunakan untuk penilaian esai otomatis di LMS.

**Pipeline:**
1. Load embedding + metadata
2. Stratified split: 85% train / 15% val (dari data asli saja)
3. Precompute 5 scalar features → normalisasi global
4. Training ensemble 3 seed (ordinal regression)
5. Evaluasi & simpan model + konfigurasi inferensi

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input, Bidirectional, LSTM, Dense, Dropout,
    Multiply, concatenate, Dot
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from sklearn.metrics import mean_absolute_error, mean_squared_error, cohen_kappa_score
from sklearn.model_selection import StratifiedShuffleSplit
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pickle
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── Konfigurasi ───────────────────────────────────────────────────────────────
DATASET_SLUG  = "siamese-data"              # nama dataset Kaggle
DATA_DIR      = f"/kaggle/input/{DATASET_SLUG}"
OUT_DIR       = "/kaggle/working"

# Hyperparameter (dari Optuna tuning pada siamese_bilstm_direct.ipynb)
BILSTM_UNITS  = 128
DROPOUT       = 0.40
EPOCHS        = 150
BATCH_SIZE    = 16
PATIENCE      = 15
LR            = 2.68e-3

# Ensemble & split
N_SEEDS       = 3
VAL_RATIO     = 0.15    # 15% data asli untuk validasi (early stopping)
RANDOM_STATE  = 42

gpus = tf.config.list_physical_devices('GPU')
print(f"TensorFlow  : {tf.__version__}")
print(f"GPU tersedia: {len(gpus)}")
for g in gpus:
    print(" ", g)

for fname in ['final_questions_emb.npy', 'final_answerkeys_emb.npy',
              'final_answers_emb.npy',    'final_metadata.pkl']:
    path   = os.path.join(DATA_DIR, fname)
    status = "OK" if os.path.exists(path) else "TIDAK DITEMUKAN"
    print(f"  {fname:<35} -> {status}")

In [ ]:
# ── Load Data ─────────────────────────────────────────────────────────────────
answers_emb    = np.load(os.path.join(DATA_DIR, 'final_answers_emb.npy'))
uniq_q_emb     = np.load(os.path.join(DATA_DIR, 'final_questions_emb.npy'))
uniq_ak_emb    = np.load(os.path.join(DATA_DIR, 'final_answerkeys_emb.npy'))
metadata       = pd.read_pickle(os.path.join(DATA_DIR, 'final_metadata.pkl'))
metadata       = metadata.reset_index(drop=True)

# Buat psj_idx jika belum ada
if 'psj_idx' not in metadata.columns:
    idpsj_sorted = sorted(metadata['IDPSJ'].unique())
    metadata['psj_idx'] = metadata['IDPSJ'].map(
        {p: i for i, p in enumerate(idpsj_sorted)}
    )

# Rekonstruksi array penuh
questions_emb  = uniq_q_emb[metadata['psj_idx'].values]
answerkeys_emb = uniq_ak_emb[metadata['psj_idx'].values]

print(f"answers_emb    : {answers_emb.shape}")
print(f"questions_emb  : {questions_emb.shape}")
print(f"answerkeys_emb : {answerkeys_emb.shape}")
print(f"metadata       : {len(metadata)} baris")
print(f"Kolom          : {list(metadata.columns)}")
print(f"\nIDPSJ unik: {sorted(metadata['IDPSJ'].unique())}")
print(f"\nDistribusi grade:")
print(metadata['grade'].value_counts().sort_index())

In [ ]:
# ── Train / Val Split ─────────────────────────────────────────────────────────
# Val: 15% data asli (stratified by grade) → digunakan untuk early stopping
# Train: 85% data asli + SEMUA data sintetis

if 'is_synthetic' in metadata.columns:
    is_real = ~metadata['is_synthetic'].values
else:
    is_real = ~metadata['IDJwb'].astype(str).str.startswith(('syn_', 'cpi_')).values

real_idx    = np.where(is_real)[0]
synth_idx   = np.where(~is_real)[0]
real_grades = metadata['grade'].values[real_idx].astype(int)

sss = StratifiedShuffleSplit(n_splits=1, test_size=VAL_RATIO, random_state=RANDOM_STATE)
for tr_rel, val_rel in sss.split(real_idx, real_grades):
    train_real_idx = real_idx[tr_rel]
    val_idx        = real_idx[val_rel]

train_idx = np.concatenate([train_real_idx, synth_idx])
y_all     = metadata['grade'].values.astype(np.float32)
y_train   = y_all[train_idx]
y_val     = y_all[val_idx]

print(f"Total data : {len(metadata)} ({is_real.sum()} asli + {(~is_real).sum()} sintetis)")
print(f"Train      : {len(train_idx)} ({len(train_real_idx)} asli + {len(synth_idx)} sintetis)")
print(f"Val        : {len(val_idx)} (data asli, stratified)")
print(f"\nDistribusi grade val:")
print(pd.Series(y_val.astype(int)).value_counts().sort_index())

In [ ]:
# ── Scalar Feature Computation ────────────────────────────────────────────────
# 5 fitur per sampel:
#   [coverage_recall, coverage_precision, coverage_f1, cos_sim_mean, length_ratio]
# Normalisasi global menggunakan statistik training → disimpan untuk inferensi.

def compute_scalar_features(answers_emb, answerkeys_emb, verbose=True):
    n     = answers_emb.shape[0]
    feats = np.zeros((n, 5), dtype=np.float32)
    if verbose:
        print(f"Menghitung scalar features untuk {n} sampel...")
    for i in range(n):
        if verbose and i % 500 == 0:
            print(f"  {i}/{n}")
        a  = answers_emb[i].astype(np.float64)
        ak = answerkeys_emb[i].astype(np.float64)

        ak_norm = ak / (np.linalg.norm(ak, axis=-1, keepdims=True) + 1e-8)
        a_norm  = a  / (np.linalg.norm(a,  axis=-1, keepdims=True) + 1e-8)

        ak_mask = np.abs(ak).sum(axis=-1) > 1e-6
        a_mask  = np.abs(a ).sum(axis=-1) > 1e-6
        ak_n    = ak_norm[ak_mask]
        a_n     = a_norm[a_mask]

        if ak_n.shape[0] == 0 or a_n.shape[0] == 0:
            continue

        sim = ak_n @ a_n.T                         # (n_ak, n_a)
        rec = float(sim.max(axis=1).mean())         # coverage recall
        pre = float(sim.max(axis=0).mean())         # coverage precision
        f1  = 2.0 * rec * pre / (rec + pre + 1e-8)

        m_ak = ak_n.mean(axis=0)
        m_a  = a_n.mean(axis=0)
        cos  = float(m_ak @ m_a / (np.linalg.norm(m_ak) * np.linalg.norm(m_a) + 1e-8))
        lrat = float(a_mask.sum()) / max(float(ak_mask.sum()), 1.0)

        feats[i] = [rec, pre, f1, cos, lrat]
    if verbose:
        print(f"  Selesai. Shape: {feats.shape}")
    return feats


all_scalar_feats = compute_scalar_features(answers_emb, answerkeys_emb)

# Statistik normalisasi dari data training → disimpan untuk inferensi
scalar_mean = all_scalar_feats[train_idx].mean(axis=0)
scalar_std  = all_scalar_feats[train_idx].std(axis=0) + 1e-8

scalar_norm = np.clip(
    (all_scalar_feats - scalar_mean) / scalar_std, -3.0, 3.0
).astype(np.float32)

N_SCALAR = all_scalar_feats.shape[1]   # 5

print(f"N_SCALAR     : {N_SCALAR}")
print(f"scalar_mean  : {scalar_mean}")
print(f"scalar_std   : {scalar_std}")

In [ ]:
# ── Custom Layers (menghindari Lambda agar model bisa disimpan/dimuat) ─────────

class AttentionPooling(tf.keras.layers.Layer):
    """Soft attention pooling atas output BiLSTM (return_sequences=True)."""

    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        self.score_dense = Dense(1, activation='tanh', use_bias=False)

    def call(self, seq_out):
        score   = self.score_dense(seq_out)             # (batch, seq, 1)
        weights = tf.nn.softmax(score, axis=1)           # (batch, seq, 1)
        return tf.reduce_sum(seq_out * weights, axis=1)  # (batch, dim)

    def get_config(self):
        return super().get_config()


class AbsDiff(tf.keras.layers.Layer):
    """Element-wise absolute difference: |a - b|."""

    def call(self, inputs):
        a, b = inputs
        return tf.abs(a - b)


class OneMinus(tf.keras.layers.Layer):
    """Orisinalitas: 1 - cos_sim_q_a."""

    def call(self, x):
        return 1.0 - x


# ── Loss & Metric ─────────────────────────────────────────────────────────────

def ordinal_loss(y_true, y_pred):
    """Sum of binary cross-entropies: P(grade > k) untuk k = 1..9."""
    thresholds = tf.cast(tf.range(1, 10), tf.float32)
    y_true_exp = tf.expand_dims(tf.cast(y_true, tf.float32), -1)
    y_binary   = tf.cast(y_true_exp > thresholds, tf.float32)
    return tf.reduce_mean(
        tf.keras.losses.binary_crossentropy(y_binary, y_pred)
    )


def ordinal_mae(y_true, y_pred):
    """Grade = jumlah threshold yang dilampaui + 1."""
    grade_pred = tf.reduce_sum(tf.cast(y_pred > 0.5, tf.float32), axis=-1) + 1.0
    return tf.reduce_mean(tf.abs(tf.cast(y_true, tf.float32) - grade_pred))


# ── Daftar custom objects (dipakai saat load_model) ───────────────────────────
CUSTOM_OBJECTS = {
    'ordinal_loss'    : ordinal_loss,
    'ordinal_mae'     : ordinal_mae,
    'AttentionPooling': AttentionPooling,
    'AbsDiff'         : AbsDiff,
    'OneMinus'        : OneMinus,
}


# ── Arsitektur Model v11 ──────────────────────────────────────────────────────

def build_model(q_seq_len, ak_seq_len, a_seq_len, emb_dim=300, n_scalar=5):
    """
    Siamese BiLSTM v11 untuk LMS.

    Input  : question, answerkey, answer (embedding), scalar features
    Output : 9 sigmoid (ordinal) → grade = sum(out > 0.5) + 1

    Merged: ea(256) + eak(256) + eq(256) + abs_diff(256) + had_prod(256)
            + cos_sim_ak_a(1) + cos_sim_q_a(1) + orisinalitas(1)
            + scalar_dense(32)  = 1315D
    Head  : Dense(512) → Dropout → Dense(64) → Dense(9, sigmoid)
    """
    shared_bilstm = Bidirectional(
        LSTM(BILSTM_UNITS, return_sequences=True), name='bilstm_shared'
    )

    inp_q      = Input(shape=(q_seq_len,  emb_dim), name='inp_q')
    inp_ak     = Input(shape=(ak_seq_len, emb_dim), name='inp_ak')
    inp_a      = Input(shape=(a_seq_len,  emb_dim), name='inp_a')
    inp_scalar = Input(shape=(n_scalar,),            name='inp_scalar')

    eq  = AttentionPooling(name='q_attn') (shared_bilstm(inp_q))
    eak = AttentionPooling(name='ak_attn')(shared_bilstm(inp_ak))
    ea  = AttentionPooling(name='a_attn') (shared_bilstm(inp_a))

    abs_diff     = AbsDiff(name='abs_diff')([eak, ea])
    had_prod     = Multiply(name='had_prod')([eak, ea])
    cos_sim_ak_a = Dot(axes=1, normalize=True, name='cos_sim_ak_a')([eak, ea])
    cos_sim_q_a  = Dot(axes=1, normalize=True, name='cos_sim_q_a')([eq, ea])
    orisinalitas = OneMinus(name='orisinalitas')(cos_sim_q_a)

    scalar_feat = Dense(32, activation='relu', name='scalar_dense')(inp_scalar)

    merged = concatenate(
        [ea, eak, eq, abs_diff, had_prod,
         cos_sim_ak_a, cos_sim_q_a, orisinalitas, scalar_feat],
        name='merged'
    )

    x   = Dense(512, activation='relu')(merged)
    x   = Dropout(DROPOUT)(x)
    x   = Dense(64,  activation='relu')(x)
    out = Dense(9,   activation='sigmoid', name='ordinal_out')(x)

    model = Model(
        inputs=[inp_q, inp_ak, inp_a, inp_scalar],
        outputs=out,
        name='siamese_bilstm_v11_lms'
    )
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
        loss=ordinal_loss,
        metrics=[ordinal_mae]
    )
    return model


_tmp = build_model(
    q_seq_len  = questions_emb.shape[1],
    ak_seq_len = answerkeys_emb.shape[1],
    a_seq_len  = answers_emb.shape[1],
    emb_dim    = answers_emb.shape[2],
    n_scalar   = N_SCALAR
)
_tmp.summary()
del _tmp

In [ ]:
# ── Training (Ensemble 3 Seed) ────────────────────────────────────────────────

def get_split(arr, idx):
    return arr[idx].astype(np.float32)

X_q_tr   = get_split(questions_emb,  train_idx)
X_ak_tr  = get_split(answerkeys_emb, train_idx)
X_a_tr   = get_split(answers_emb,    train_idx)
X_s_tr   = scalar_norm[train_idx]

X_q_val  = get_split(questions_emb,  val_idx)
X_ak_val = get_split(answerkeys_emb, val_idx)
X_a_val  = get_split(answers_emb,    val_idx)
X_s_val  = scalar_norm[val_idx]

# Sample weights: inverse class frequency
grade_int          = y_train.astype(int)
unique_g, counts_g = np.unique(grade_int, return_counts=True)
freq_map           = dict(zip(unique_g, counts_g))
raw_w              = np.array([len(y_train) / (len(unique_g) * freq_map[g]) for g in grade_int])
sample_w           = (raw_w / raw_w.mean()).astype(np.float32)

print(f"Sample weight: min={sample_w.min():.3f}  max={sample_w.max():.3f}  mean={sample_w.mean():.3f}")

saved_model_paths = []

for seed in range(N_SEEDS):
    print(f"\n{'='*55}")
    print(f"Seed {seed + 1}/{N_SEEDS}")
    print(f"{'='*55}")

    tf.random.set_seed(seed)
    np.random.seed(seed)

    model = build_model(
        q_seq_len  = questions_emb.shape[1],
        ak_seq_len = answerkeys_emb.shape[1],
        a_seq_len  = answers_emb.shape[1],
        emb_dim    = answers_emb.shape[2],
        n_scalar   = N_SCALAR
    )

    model_path = os.path.join(OUT_DIR, f'lms_model_seed{seed}.keras')

    callbacks = [
        EarlyStopping(
            monitor='val_ordinal_mae', patience=PATIENCE,
            restore_best_weights=True, mode='min', verbose=1
        ),
        ReduceLROnPlateau(
            monitor='val_ordinal_mae', factor=0.5, patience=5,
            min_lr=1e-6, mode='min', verbose=0
        ),
        ModelCheckpoint(
            model_path, monitor='val_ordinal_mae',
            save_best_only=True, mode='min', verbose=0
        ),
    ]

    model.fit(
        [X_q_tr, X_ak_tr, X_a_tr, X_s_tr], y_train,
        sample_weight=sample_w,
        validation_data=([X_q_val, X_ak_val, X_a_val, X_s_val], y_val),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=callbacks, verbose=1
    )

    saved_model_paths.append(model_path)
    print(f"  Disimpan -> {model_path}")
    tf.keras.backend.clear_session()

print(f"\nTraining selesai. {N_SEEDS} model disimpan.")

In [ ]:
# ── Evaluasi pada Val Set ─────────────────────────────────────────────────────

loaded_models = [
    tf.keras.models.load_model(p, custom_objects=CUSTOM_OBJECTS)
    for p in saved_model_paths
]

all_sigmoid = [
    m.predict([X_q_val, X_ak_val, X_a_val, X_s_val], verbose=0)
    for m in loaded_models
]
mean_sigmoid = np.mean(all_sigmoid, axis=0)
y_pred_val   = np.clip(
    np.sum(mean_sigmoid > 0.5, axis=-1) + 1, 1, 10
).astype(np.float32)

mae_val  = mean_absolute_error(y_val, y_pred_val)
rmse_val = np.sqrt(mean_squared_error(y_val, y_pred_val))
qwk_val  = cohen_kappa_score(y_val.astype(int), y_pred_val.astype(int),
                              weights='quadratic')

print(f"Evaluasi pada Val Set ({len(y_val)} sampel asli)")
print(f"  MAE  : {mae_val:.4f}")
print(f"  RMSE : {rmse_val:.4f}")
print(f"  QWK  : {qwk_val:.4f}")

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.scatter(y_val, y_pred_val, alpha=0.4, edgecolors='k', linewidths=0.3)
ax.plot([1, 10], [1, 10], 'r--', label='Ideal')
ax.set_xlabel('Grade Aktual')
ax.set_ylabel('Grade Prediksi')
ax.set_title(f'Prediksi vs Aktual (Val Set)\nMAE={mae_val:.4f}  RMSE={rmse_val:.4f}  QWK={qwk_val:.4f}')
ax.set_xticks(range(1, 11))
ax.set_yticks(range(1, 11))
ax.legend()

ax2 = axes[1]
bins = np.arange(0.5, 11.5, 1)
ax2.hist(y_val,      bins=bins, alpha=0.6, color='steelblue',  label='Aktual',   density=True)
ax2.hist(y_pred_val, bins=bins, alpha=0.6, color='darkorange', label='Prediksi', density=True)
ax2.axvline(y_val.mean(),      color='blue',   linestyle='--', linewidth=1.5,
            label=f'\u03bc aktual={y_val.mean():.2f}')
ax2.axvline(y_pred_val.mean(), color='orange', linestyle='--', linewidth=1.5,
            label=f'\u03bc prediksi={y_pred_val.mean():.2f}')
ax2.set_xlabel('Grade')
ax2.set_ylabel('Density')
ax2.set_title('Distribusi Grade Aktual vs Prediksi')
ax2.legend()

plt.tight_layout()
plot_path = os.path.join(OUT_DIR, 'lms_eval.png')
plt.savefig(plot_path, dpi=150)
plt.show()
print(f"Plot disimpan -> {plot_path}")

In [ ]:
# ── Simpan Konfigurasi Inferensi ──────────────────────────────────────────────

inference_config = {
    # Normalisasi scalar features
    'scalar_mean'  : scalar_mean,
    'scalar_std'   : scalar_std,

    # Nama file model (relatif, tanpa path)
    'model_paths'  : [os.path.basename(p) for p in saved_model_paths],

    # Dimensi input
    'q_seq_len'    : int(questions_emb.shape[1]),
    'ak_seq_len'   : int(answerkeys_emb.shape[1]),
    'a_seq_len'    : int(answers_emb.shape[1]),
    'emb_dim'      : int(answers_emb.shape[2]),
    'n_scalar'     : N_SCALAR,

    # Metadata
    'val_mae'      : float(mae_val),
    'val_rmse'     : float(rmse_val),
    'val_qwk'      : float(qwk_val),
    'n_train'      : int(len(train_idx)),
    'n_val'        : int(len(val_idx)),
    'bilstm_units' : BILSTM_UNITS,
    'n_seeds'      : N_SEEDS,
}

config_path = os.path.join(OUT_DIR, 'lms_inference_config.pkl')
with open(config_path, 'wb') as f:
    pickle.dump(inference_config, f)

print("File yang disimpan:")
for p in saved_model_paths:
    print(f"  Model  : {p}")
print(f"  Config : {config_path}")
print()
print(f"Val MAE={mae_val:.4f}  RMSE={rmse_val:.4f}  QWK={qwk_val:.4f}")

In [ ]:
# ── Fungsi Inferensi untuk LMS ────────────────────────────────────────────────
# Salin fungsi compute_scalar_features, custom layer classes, CUSTOM_OBJECTS,
# dan predict_grade ke script deployment LMS.

def predict_grade(questions_emb_new, answerkeys_emb_new, answers_emb_new,
                  models, scalar_mean, scalar_std,
                  q_seq_len, ak_seq_len, a_seq_len):
    """
    Prediksi nilai esai baru.

    Parameter:
        questions_emb_new   : np.ndarray (n, q_seq_len, 300)
        answerkeys_emb_new  : np.ndarray (n, ak_seq_len, 300)
        answers_emb_new     : np.ndarray (n, a_seq_len, 300)
        models              : list[tf.keras.Model]
        scalar_mean, std    : np.ndarray (5,)  — dari lms_inference_config.pkl
        q/ak/a_seq_len      : int              — dari lms_inference_config.pkl

    Return:
        grades       : np.ndarray (n,) int  — nilai [1-10]
        sigmoid_prob : np.ndarray (n, 9)    — probabilitas ordinal
    """
    def pad_or_crop(arr, target_len):
        n, cur_len, dim = arr.shape
        if cur_len == target_len:
            return arr
        if cur_len < target_len:
            pad = np.zeros((n, target_len - cur_len, dim), dtype=arr.dtype)
            return np.concatenate([arr, pad], axis=1)
        return arr[:, :target_len, :]

    X_q  = pad_or_crop(questions_emb_new.astype(np.float32),  q_seq_len)
    X_ak = pad_or_crop(answerkeys_emb_new.astype(np.float32), ak_seq_len)
    X_a  = pad_or_crop(answers_emb_new.astype(np.float32),    a_seq_len)

    feats = compute_scalar_features(X_a, X_ak, verbose=False)
    X_s   = np.clip((feats - scalar_mean) / scalar_std, -3.0, 3.0).astype(np.float32)

    all_sigmoid  = [m.predict([X_q, X_ak, X_a, X_s], verbose=0) for m in models]
    mean_sigmoid = np.mean(all_sigmoid, axis=0)
    grades       = np.clip(np.sum(mean_sigmoid > 0.5, axis=-1) + 1, 1, 10).astype(int)
    return grades, mean_sigmoid


# ── Template memuat model di LMS ─────────────────────────────────────────────
print("""Template memuat model di backend LMS:

import pickle, tensorflow as tf
# (salin juga: AttentionPooling, AbsDiff, OneMinus, CUSTOM_OBJECTS,
#              ordinal_loss, ordinal_mae, compute_scalar_features, predict_grade)

with open('lms_inference_config.pkl', 'rb') as f:
    cfg = pickle.load(f)

models = [tf.keras.models.load_model(p, custom_objects=CUSTOM_OBJECTS)
          for p in cfg['model_paths']]

grades, probs = predict_grade(
    q_emb, ak_emb, a_emb, models,
    cfg['scalar_mean'], cfg['scalar_std'],
    cfg['q_seq_len'], cfg['ak_seq_len'], cfg['a_seq_len']
)
""")